# D4 — The walk operator **is** attention / El operador de walk **es** atención

🇬🇧 This is the central idea of the whole project. We will build it in three small steps:
1. Recall how a Transformer's **attention** works (one line).
2. Notice that aggregating over length-`g` walks has the *exact same shape*.
3. Conclude: the path algebra gives us a **sparse, multi-hop attention** — and that is what beats over-squashing.

🇪🇸 Esta es la idea central de todo el proyecto. La construiremos en tres pasos pequeños:
1. Recordar cómo funciona la **atención** de un Transformer (una línea).
2. Notar que agregar sobre recorridos de longitud `g` tiene *exactamente la misma forma*.
3. Concluir: el álgebra de caminos nos da una **atención sparse y multi-salto** — y eso es lo que vence al
   over-squashing.

## Step 1 — attention in one line / Paso 1 — atención en una línea

🇬🇧 A Transformer updates token `j` by a **weighted sum** of all other tokens' values:
$$h_j = \sum_i \alpha_{ji}\, v_i, \qquad \alpha_{ji} = \text{softmax}_i\big(q_j \cdot k_i\big).$$
The weights `α` are **learned** from content (query · key). Crucially, standard attention is **all-pairs**:
every token may attend to every other.

🇪🇸 Un Transformer actualiza el token `j` con una **suma ponderada** de los valores de todos los demás:
$$h_j = \sum_i \alpha_{ji}\, v_i, \qquad \alpha_{ji} = \text{softmax}_i\big(q_j \cdot k_i\big).$$
Los pesos `α` se **aprenden** del contenido (query · key). Y algo clave: la atención estándar es
**todos-contra-todos**: cada token puede atender a cualquier otro.

## Step 2 — the same shape, but over walks / Paso 2 — la misma forma, pero sobre recorridos

🇬🇧 Now look at how a node aggregates information from `g` hops away. It is also a weighted sum over other
nodes:
$$h_j = \sum_i W^g_{ji}\, (\text{message from } i).$$
The only questions are: **(a) which `i` are included**, and **(b) what are the weights `W`**. The path algebra
answers (a): node `i` is included exactly when there is a length-`g` walk `i → j` — i.e. when `n_g(i,j) > 0`.
That is a **mask**.

🇪🇸 Ahora mira cómo un nodo agrega información de `g` saltos de distancia. También es una suma ponderada sobre
otros nodos:
$$h_j = \sum_i W^g_{ji}\, (\text{mensaje de } i).$$
Las únicas preguntas son: **(a) qué `i` se incluyen**, y **(b) cuáles son los pesos `W`**. El álgebra de
caminos responde (a): el nodo `i` se incluye exactamente cuando hay un recorrido de longitud `g` `i → j` —
es decir, cuando `n_g(i,j) > 0`. Eso es una **máscara**.

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt
from oversquash.data_bottleneck import make_bottleneck_graph
from oversquash.ideal_bridge import walk_operators

data = make_bottleneck_graph(5, 4, 3, torch.Generator().manual_seed(0))
ei, N = data.edge_index.numpy(), data.num_nodes
raw, _ = walk_operators(ei, N, max_length=4)

# the attention SUPPORT at range g = where walks exist / el SOPORTE de atención en rango g
fig, axes = plt.subplots(1, 4, figsize=(13, 3.4))
for g in range(4):
    mask = (raw[g] > 0).astype(int)
    axes[g].imshow(mask, cmap='Greens', vmin=0, vmax=1)
    axes[g].set_title(f'range g={g+1}\n{mask.sum()} / {N*N} pairs')
    axes[g].set_xlabel('source i'); axes[g].set_ylabel('target j')
plt.suptitle('the path-reachability MASK = the attention support (sparse!)')
plt.tight_layout(); plt.show()

🇬🇧 Notice how **sparse** these masks are — mostly white (no connection). Standard attention would fill the
whole square (all-pairs). The path algebra tells us we only need the green entries — at the deepest range,
that is about **2%** of the full square. We get *global reach* (multi-hop) at a *tiny fraction* of the cost.

🇪🇸 Fíjate qué **dispersas** (sparse) son estas máscaras — casi todo blanco (sin conexión). La atención
estándar llenaría todo el cuadrado (todos-contra-todos). El álgebra de caminos nos dice que solo necesitamos
las entradas verdes — en el rango más profundo, eso es cerca del **2%** del cuadrado completo. Obtenemos
*alcance global* (multi-salto) a una *fracción mínima* del costo.

## Step 3 — fixed weights vs. learned weights / Paso 3 — pesos fijos vs. pesos aprendidos

🇬🇧 For (b), the weights, there are two choices:
- **Fixed** `W = n_g(i,j)` (the raw path count). This is the `walkraw` model. It reaches the target, but it
  weights *every* reachable source the same — it cannot say *which* one matters.
- **Learned** `W = α_{ji}` from query·key, computed **only over the masked pairs**. This is `WalkAttention`.
  It is exactly Transformer attention, restricted to walk-reachable pairs.

Below: same support, the two weightings side by side.

🇪🇸 Para (b), los pesos, hay dos opciones:
- **Fijos** `W = n_g(i,j)` (el conteo crudo de caminos). Es el modelo `walkraw`. Alcanza el objetivo, pero
  pesa *todas* las fuentes alcanzables igual — no puede decir *cuál* importa.
- **Aprendidos** `W = α_{ji}` de query·key, calculados **solo sobre los pares de la máscara**. Es
  `WalkAttention`. Es exactamente la atención de un Transformer, restringida a los pares alcanzables por
  recorridos.

Abajo: el mismo soporte, las dos ponderaciones lado a lado.

In [ ]:
from oversquash.attention import WalkAttention\nfrom torch_geometric.utils import softmax as pyg_softmax\ng = 3  # deepest range / rango más profundo\nt = int(data.root_mask.nonzero()[0])\n\n# Build the mask the SAME way the real layer does: transpose so indices()[0]=target,\n# indices()[1]=source (mirrors WalkAttention.forward exactly).\n# Construimos la máscara como lo hace la capa real: transpuesta, así indices()[0]=objetivo,\n# indices()[1]=fuente (refleja WalkAttention.forward exactamente).\nmask = torch.from_numpy((raw[g] > 0).T.astype('float32')).to_sparse().coalesce()\ntgt, src = mask.indices()[0], mask.indices()[1]\n\n# FIXED weights = raw path counts (what walkraw uses) / pesos FIJOS = conteos crudos\nWfixed = raw[g][src.numpy(), tgt.numpy()]   # n_g(src -> tgt)\n\n# LEARNED weights from an (untrained) WalkAttention layer, to show they DIFFER per source\n# pesos APRENDIDOS de una capa WalkAttention (sin entrenar), para ver que DIFIEREN por fuente\ntorch.manual_seed(0)\nlayer = WalkAttention(data.x.size(1), 8, n_heads=1)\nq = layer.q(data.x).view(N,1,8); k = layer.k(data.x).view(N,1,8)\nlogit = (q[tgt]*k[src]).sum(-1) * layer.scale\nalpha = pyg_softmax(logit, tgt, num_nodes=N).detach().numpy().ravel()\n\ninto_t = (tgt.numpy() == t)\nprint(f'Sources that reach target node {t}: {src.numpy()[into_t]}')\nprint(f'  FIXED   (walkraw):   {Wfixed[into_t].round(2)}   <- ALL EQUAL: cannot select')\nprint(f'  LEARNED (attention): {alpha[into_t].round(3)}   <- DIFFERENT per source: can select')

## The one-sentence summary / El resumen en una frase

🇬🇧 **The path algebra says WHO can attend to whom across many hops (a sparse mask); attention learns HOW MUCH
each one matters.** Fixed path-counts can't tell sources apart; learned attention can. That single difference
turns a partial fix into a complete one — which we prove with real training in **D5**.

🇪🇸 **El álgebra de caminos dice QUIÉN puede atender a quién a través de muchos saltos (una máscara sparse); la
atención aprende CUÁNTO importa cada uno.** Los conteos fijos no distinguen fuentes; la atención aprendida sí.
Esa única diferencia convierte un arreglo parcial en uno completo — lo cual demostramos con entrenamiento real
en **D5**.